## Planning:
- Load data
- Create sample FE - sum of boolean and proportion of boolean
- Create TE in run.cv
- Implement for XGBoost

In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
submit = pd.read_csv("data/sample_submission.csv")

In [3]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()
numeric_cols_ex = numeric_cols.copy()
numeric_cols_ex.remove("is_fraud")

In [4]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [5]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

In [6]:
for col in obj_cols:
    counts = train.groupby(col)["id"].count()
    proportion = counts/counts.sum()

    print(f'col: {col}')
    summary = pd.DataFrame({'count': counts, 'proportion': proportion})
    print(summary)
    print('-'*20)

col: merchant_category
                    count  proportion
merchant_category                    
Crypto Exchange      5285    0.007550
Electronics         48693    0.069565
Fuel                59880    0.085548
Gaming              23784    0.033979
Gift Cards           9456    0.013509
Groceries          142578    0.203694
Healthcare          35889    0.051273
Online Retail      155097    0.221580
Restaurants        101360    0.144808
Streaming           47196    0.067427
Travel              28715    0.041024
Utilities           42027    0.060042
--------------------
col: card_type
             count  proportion
card_type                     
Amex         45797    0.065428
Discover     17235    0.024623
Mastercard  254857    0.364102
RuPay        37070    0.052960
Visa        345001    0.492887
--------------------
col: auth_method
                    count  proportion
auth_method                          
3D Secure          210036    0.300069
Biometric           93666    0.133816
No

## Model using XGboost + TE

In [7]:
from sklearn.preprocessing import TargetEncoder

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [10]:
import xgboost as xgb

In [11]:
X = train.drop(columns=["id","is_fraud"]).copy()
y = train["is_fraud"].copy()

### Insert TE in the run_cv loop

In [12]:
def run_cv(fit_predict_fn, te_cols, name): # adding input te_cols for categorical variable required for TE
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        # splitting into training and evaluation
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y[tr_idx].copy(), y[va_idx].copy()
        # --- FIX: Fill NaNs and ensure string type for TE columns ---
        #for col in te_cols:
        #    X_tr.loc[:,col] = X_tr.loc[:,col].fillna("Missing").astype(str)
        #   X_va.loc[:,col] = X_va.loc[:,col].fillna("Missing").astype(str)
        #   X_test.loc[:,col] = X_test.loc[:,col].fillna("Missing").astype(str)
        
        # -----------------------------------------------------------
        # creating TE model for training data
        te = TargetEncoder(smooth="auto",target_type='binary')
        
        # fit on tr data
        te.fit(X_tr[te_cols],y_tr)

        # transform on tr data
        X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])

        # transform on va data
        X_va.loc[:, te_cols] = te.transform(X_va[te_cols])
        
        # transform on X_test data
        X_test = test.drop(columns=["id"]).copy()
        # X_test_obj = X_test[obj_cols].copy()
        X_test.loc[:, te_cols] = te.transform(X_test[obj_cols])

        # print(f'x_test',X_test.head(2))

        # print(f'X_tr_col: ',X_tr.info())
        for col in te_cols:
            X_tr[col] = X_tr[col].astype('float64')
            X_va[col] = X_va[col].astype('float64')
            X_test[col] = X_test[col].astype('float64')
            # X_tr.loc[:,col] = X_tr.loc[:,col].astype('float64')
            # X_va.loc[:,col] = X_va.loc[:,col].astype('float64')
            # X_test.loc[:,col] = X_test.loc[:,col].astype('float64')

        # print(f'x_tr ', X_tr.info())
        # print(f'x_va ', X_va.info())
        # print(f'x_test ', X_test.info())
        # start predit
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va,X_test)

        # save prediction in oof and in test
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [13]:
def fit_xgb(X_tr, y_tr, X_va, y_va,X_test):
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=10000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20,
        colsample_bytree=0.8,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method='hist',
        enable_categorical=True,
        early_stopping_rounds=200,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [14]:
def save_sub(pred_1,pred_2, name_1,name_2):
    sub = submit.copy()
    sub['is_fraud'] = pred_1
    sub.to_csv(name_1, index=False)
    print('save:', name_1)

    oof_df = train[['id']].copy()
    oof_df.loc[:,name_2] = oof_xgb
    oof_df.to_csv(name_2,index=False)
    print('save:', name_2)

In [15]:
seed = [42,67,1508,2312,106]

In [16]:
auc_xgb_dict = {}

In [17]:
for Seed in seed:
    SEED = Seed
    N_FOLDS = 5
    TARGET = 'is_fraud'

    # Stratified KFold cross validation
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    FOLDS = list(skf.split(X, y))

    oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, obj_cols, 'XGBoost')
    save_sub(test_xgb, oof_xgb, f'sub/sub_xgb_te_{SEED}.csv', f'oof/oof_xgb_te_{SEED}.csv')
    auc_xgb_dict[Seed] = auc_xgb

  fold 0: AUC = 0.94458
  fold 1: AUC = 0.94845
  fold 2: AUC = 0.95120
  fold 3: AUC = 0.94602
  fold 4: AUC = 0.94127
XGBoost | OOF AUC = 0.94625 (mean 0.94631 ± 0.00338)
save: sub/sub_xgb_te_42.csv
save: oof/oof_xgb_te_42.csv
  fold 0: AUC = 0.94715
  fold 1: AUC = 0.94698
  fold 2: AUC = 0.94451
  fold 3: AUC = 0.94708
  fold 4: AUC = 0.94647
XGBoost | OOF AUC = 0.94641 (mean 0.94644 ± 0.00099)
save: sub/sub_xgb_te_67.csv
save: oof/oof_xgb_te_67.csv
  fold 0: AUC = 0.94445
  fold 1: AUC = 0.94775
  fold 2: AUC = 0.94871
  fold 3: AUC = 0.94683
  fold 4: AUC = 0.94415
XGBoost | OOF AUC = 0.94637 (mean 0.94638 ± 0.00180)
save: sub/sub_xgb_te_1508.csv
save: oof/oof_xgb_te_1508.csv
  fold 0: AUC = 0.94979
  fold 1: AUC = 0.95152
  fold 2: AUC = 0.94416
  fold 3: AUC = 0.94430
  fold 4: AUC = 0.94229
XGBoost | OOF AUC = 0.94632 (mean 0.94641 ± 0.00358)
save: sub/sub_xgb_te_2312.csv
save: oof/oof_xgb_te_2312.csv
  fold 0: AUC = 0.94684
  fold 1: AUC = 0.94781
  fold 2: AUC = 0.94689
  fo

In [18]:
auc_xgb_dict

{42: 0.9462501224825093,
 67: 0.9464137801350211,
 1508: 0.9463692460013123,
 2312: 0.9463207138313678,
 106: 0.9464315994564808}